# Проверка моделей из `artifacts/`

Ожидаются файлы `classification_{ax|sa|co}_model.pt` и `segmentation_{ax|sa|co}_model.pt`.  
Изображение — из `data/segmentation_task/.../test/images/` (рядом маска в `test/masks/`).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torchvision.transforms as T
from PIL import Image

from main import CLASSES, SimpleUNet, SmallResNet, device

ARTIFACTS = Path("artifacts")
DATA_SEG = Path("data/segmentation_task")

_cls_cache: dict[str, tuple] = {}
_seg_cache: dict[str, tuple] = {}


def _parse_true_class_from_filename(name: str) -> str:
    parts = Path(name).stem.split("_")
    code = parts[3] if len(parts) > 3 else ""
    return {"me": "meningioma", "gl": "glioma", "pi": "pituitary", "no": "no_tumor"}.get(code, "?")


def _load_classification(plane: str):
    if plane in _cls_cache:
        return _cls_cache[plane]
    path = ARTIFACTS / f"classification_{plane}_model.pt"
    ckpt = torch.load(path, map_location=device, weights_only=False)
    model = SmallResNet(num_classes=len(ckpt.get("classes", CLASSES))).to(device)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()
    _cls_cache[plane] = (model, ckpt)
    return model, ckpt


def _load_segmentation(plane: str):
    if plane in _seg_cache:
        return _seg_cache[plane]
    path = ARTIFACTS / f"segmentation_{plane}_model.pt"
    ckpt = torch.load(path, map_location=device, weights_only=False)
    model = SimpleUNet(in_ch=1, base=32, out_ch=1).to(device)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()
    _seg_cache[plane] = (model, ckpt)
    return model, ckpt


def show_prediction(image_path: str | Path, plane: str) -> None:
    """
    image_path — путь к .jpg из test/images.
    plane — 'ax' | 'sa' | 'co': выбор пар моделей из artifacts/.
    """
    plane = plane.lower().strip()
    if plane not in {"ax", "sa", "co"}:
        raise ValueError("plane должен быть ax, sa или co")

    image_path = Path(image_path)
    if not image_path.is_file():
        raise FileNotFoundError(image_path)

    true_class = _parse_true_class_from_filename(image_path.name)

    mask_path = image_path.parent.parent / "masks" / f"{image_path.stem}.png"
    if not mask_path.is_file():
        raise FileNotFoundError(f"Нет маски: {mask_path}")

    cls_model, cls_ckpt = _load_classification(plane)
    seg_model, seg_ckpt = _load_segmentation(plane)

    c_mean, c_std = cls_ckpt["mean"], cls_ckpt["std"]
    c_size = int(cls_ckpt["image_size"])
    classes_ckpt = cls_ckpt.get("classes", CLASSES)
    if isinstance(classes_ckpt, tuple):
        classes_ckpt = list(classes_ckpt)

    cls_tf = T.Compose([
        T.Resize((c_size, c_size)),
        T.ToTensor(),
        T.Normalize(mean=[c_mean], std=[c_std]),
    ])

    s_mean, s_std = seg_ckpt["mean"], seg_ckpt["std"]
    s_size = int(seg_ckpt["image_size"])

    seg_tf = T.Compose([
        T.Resize((s_size, s_size)),
        T.ToTensor(),
        T.Normalize(mean=[s_mean], std=[s_std]),
    ])

    pil = Image.open(image_path).convert("L")
    x_cls = cls_tf(pil).unsqueeze(0).to(device)
    x_seg = seg_tf(pil).unsqueeze(0).to(device)

    with torch.no_grad():
        logits_c = cls_model(x_cls)
        pred_i = logits_c.argmax(dim=1).item()
        pred_class = classes_ckpt[pred_i]

        logits_s = seg_model(x_seg)
        pred_mask = (torch.sigmoid(logits_s)[0, 0].cpu().numpy() > 0.5).astype(np.float32)

    gt_mask = np.array(Image.open(mask_path).convert("L").resize((s_size, s_size), Image.NEAREST))
    gt_mask = (gt_mask > 0).astype(np.float32)

    img_show = np.array(pil.resize((s_size, s_size), Image.BILINEAR))

    ok_cls = pred_class == true_class
    fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
    axes[0].imshow(img_show, cmap="gray")
    axes[0].set_title(f"{plane}  {image_path.name}", fontsize=9)
    axes[0].axis("off")

    axes[1].text(0.1, 0.55, f"pred: {pred_class}", fontsize=12)
    axes[1].text(0.1, 0.35, f"true: {true_class}", fontsize=12)
    axes[1].text(
        0.1, 0.12, "OK" if ok_cls else "FAIL", fontsize=14, color="green" if ok_cls else "red"
    )
    axes[1].axis("off")

    axes[2].imshow(pred_mask, cmap="gray", vmin=0, vmax=1)
    axes[2].set_title("seg pred")
    axes[2].axis("off")

    axes[3].imshow(gt_mask, cmap="gray", vmin=0, vmax=1)
    axes[3].set_title("mask GT")
    axes[3].axis("off")

    plt.tight_layout()
    plt.show()

    print(f"Класс: pred={pred_class}  true={true_class}  {'OK' if ok_cls else 'FAIL'}")

In [ ]:
# пример (подставь свой файл из test/images):
# show_prediction(DATA_SEG / "ax" / "test" / "images" / "some_file.jpg", "ax")